# PaperMind â€” ByT5 OCR Correction (A100 Full Pipeline)

**One notebook, end to end:**
1. Generate 50,000 `(noisy_trocr_output, clean_text)` pairs using real TrOCR on this A100
2. Fine-tune **`google/byt5-base`** (580M params â€” upgraded from small because A100 has 40GB VRAM)
3. Evaluate CER + exact match, qualitative check on real PaperMind errors
4. Save checkpoint to Google Drive â†’ integrate into backend

**Why ByT5-base over small on A100?**
- 2Ã— parameters â†’ better generalisation to unseen error patterns and real handwriting
- Fits in A100 40GB comfortably at batch 64 with bf16
- On T4 this would OOM â€” A100 makes it viable

**Estimated total time on A100: ~60â€“80 minutes** (vs 3.5h on T4)

**Before running:** Runtime â†’ Change runtime type â†’ **A100**

In [ ]:
# -- Cell 1: Install dependencies --
# Pin tokenizers to a version with pre-built Linux wheels
# This avoids the 'Building wheel for tokenizers (pyproject.toml)' Rust error on Colab
!pip install -q tokenizers==0.19.1
!pip install -q transformers==4.44.2 datasets evaluate sacrebleu sentencepiece accelerate Pillow
print('Done.')

In [ ]:
# â”€â”€ Cell 2: CONFIG â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# A100 settings â€” tuned for 40GB VRAM

# Dataset
N_PAIRS       = 50_000   # 50k pairs â€” A100 generates these in ~35 min
TROCR_BATCH   = 64       # TrOCR inference batch (A100 handles 64 comfortably)
CLEAN_FRAC    = 0.15     # fraction of ed=0 pairs kept (model must not over-correct)

# Model â€” UPGRADED to byt5-base on A100
BYT5_MODEL    = 'google/byt5-base'   # 580M params (use 'google/byt5-small' if you hit OOM)
MAX_LEN       = 256

# Training
EPOCHS              = 5
BYT5_TRAIN_BATCH    = 64    # A100 handles bs=64 for byt5-base with bf16
BYT5_EVAL_BATCH     = 128
LEARNING_RATE       = 4e-4  # slightly lower for base model
WARMUP_STEPS        = 300
GRAD_ACCUM_STEPS    = 1     # no need for accumulation â€” batch is already large

# Drive paths
DRIVE_DATA_DIR = '/content/drive/MyDrive/papermind/correction'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/papermind/byt5-base-correction-checkpoint'

SEED = 42

print(f'Config: N_PAIRS={N_PAIRS:,}  MODEL={BYT5_MODEL}  EPOCHS={EPOCHS}')
print(f'Batch: TrOCR={TROCR_BATCH}  Train={BYT5_TRAIN_BATCH}  Eval={BYT5_EVAL_BATCH}')

In [ ]:
# â”€â”€ Cell 3: Mount Drive + verify A100 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os, torch, math, random, time
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

assert torch.cuda.is_available(), 'No GPU â€” change runtime to A100!'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU  : {gpu_name}')
print(f'VRAM : {vram_gb:.1f} GB')

if 'A100' not in gpu_name:
    print(f'WARNING: Got {gpu_name}, not A100.')
    print('You may need to lower TROCR_BATCH to 32 and BYT5_TRAIN_BATCH to 32.')
else:
    print('A100 confirmed. All batch sizes are safe.')

In [ ]:
# â”€â”€ Cell 4: Academic corpus â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Covers all PaperMind exam sub-domains: OOP, SE, AI/ML, Cybersecurity.
# Includes the highest-error-rate patterns: question markers + option letters.
random.seed(SEED)

_MARKERS = ['1)', '2)', '3)', '4)', '5)', '6)',
            'Q1.', 'Q2.', 'Q3.', 'Q4.',
            'Ans 1:', 'Ans 2:', 'Ans 3:']

_OOP = [
    'Encapsulation bundles data and methods into a single class.',
    'Inheritance allows a subclass to reuse the behaviour of a superclass.',
    'Polymorphism enables objects of different classes to respond to the same method call.',
    'Abstraction hides implementation details and exposes only the interface.',
    'The Liskov Substitution Principle: subclass objects should replace superclass objects.',
    'A constructor is invoked automatically when an object is created.',
    'The final keyword prevents a class from being subclassed.',
    'Method overloading: same name, different parameter lists.',
    'Method overriding: subclass provides a different implementation of a superclass method.',
    'An interface defines a contract that implementing classes must fulfil.',
    'Abstract classes cannot be instantiated directly.',
    'The singleton pattern ensures only one instance of a class exists at runtime.',
    'Composition is preferred over inheritance to reduce coupling.',
    'Access modifiers: public, private, protected, and default.',
    'Static members belong to the class rather than to any instance.',
    'A stack uses Last In First Out ordering.',
    'A queue uses First In First Out ordering.',
    'Binary search runs in O(log n) time on a sorted array.',
    'Quicksort has an average time complexity of O(n log n).',
    'The factory method pattern defines an interface for creating objects.',
    'The observer pattern defines a one-to-many dependency between objects.',
    'Dependency injection reduces tight coupling between components.',
    'The decorator pattern attaches additional responsibilities dynamically.',
    'Garbage collection automatically reclaims memory from unused objects.',
    'Synchronisation prevents race conditions in multithreaded programs.',
    'A linked list stores elements in nodes connected by pointers.',
    'The open-closed principle: open for extension, closed for modification.',
    'Cohesion refers to how closely related the responsibilities of a module are.',
    'Coupling measures the degree of dependency between modules.',
    'Polymorphism can be achieved through method overriding or interfaces.',
]

_SE = [
    'Agile methodology emphasises iterative development and customer collaboration.',
    'The waterfall model is a linear sequential approach to software development.',
    'Regression testing verifies that new changes have not broken existing functionality.',
    'Continuous integration merges code changes into a shared repository frequently.',
    'SOLID: Single responsibility, Open-closed, Liskov, Interface segregation, Dependency inversion.',
    'A sprint in Scrum is a time-boxed iteration typically lasting two weeks.',
    'Test-driven development writes tests before writing the implementation code.',
    'Static analysis tools detect potential bugs without executing the program.',
    'Cyclomatic complexity measures the number of linearly independent paths through code.',
    'Technical debt accumulates when shortcuts are taken instead of proper solutions.',
    'The product backlog is an ordered list of features for the product.',
    'Code review improves quality and spreads knowledge across the team.',
    'A deployment pipeline automates building, testing, and releasing software.',
    'Version control systems track changes to source code over time.',
    'Black-box testing checks inputs and outputs without examining internal code.',
]

_AI = [
    'Supervised learning uses labelled data to map inputs to outputs.',
    'Gradient descent minimises a loss function by iteratively updating parameters.',
    'Overfitting occurs when a model learns training noise rather than the true pattern.',
    'A convolutional neural network is well suited for image recognition tasks.',
    'The attention mechanism allows the model to focus on relevant input positions.',
    'Precision: fraction of predicted positives that are actually positive.',
    'Recall: fraction of actual positives correctly identified by the model.',
    'The F1-score is the harmonic mean of precision and recall.',
    'K-means partitions data into k clusters by minimising intra-cluster distance.',
    'Transfer learning reuses a pretrained model for a different related task.',
    'A decision tree splits data on feature values to make predictions.',
    'Random forests aggregate many decision trees to reduce variance.',
    'Regularisation techniques such as L1 and L2 reduce overfitting.',
    'The activation function introduces non-linearity into a neural network.',
    'Backpropagation computes gradients by applying the chain rule layer by layer.',
]

_CYBER = [
    'Symmetric encryption uses the same key for encryption and decryption.',
    'Asymmetric encryption uses a public key to encrypt and a private key to decrypt.',
    'SQL injection inserts malicious SQL into a query to manipulate the database.',
    'Cross-site scripting injects malicious scripts into pages viewed by other users.',
    'Two-factor authentication requires two independent forms of identification.',
    'Hashing converts data into a fixed-length digest that cannot be reversed.',
    'The principle of least privilege grants only the minimum access required.',
    'A firewall monitors and controls incoming and outgoing network traffic.',
    'A man-in-the-middle attack intercepts communication between two parties.',
    'Zero-day vulnerabilities are unknown to the software vendor at the time of exploit.',
]

_SHORT = [
    'True', 'False',
    '(a) Encapsulation', '(b) Inheritance', '(c) Polymorphism', '(d) Abstraction',
    '(a) True', '(b) False',
    '(a) O(1)', '(b) O(log n)', '(c) O(n)', '(d) O(n log n)', '(e) O(n^2)',
    '(a) Stack', '(b) Queue', '(c) Heap', '(d) Tree',
    'True, encapsulation hides internal state from outside access.',
    'False, interfaces cannot contain instance variables.',
    'True, polymorphism enables runtime method dispatch.',
    'False, an abstract class may have concrete methods.',
    'True, static methods can be called without creating an instance.',
    'False, constructors are not inherited by subclasses.',
    '(a) Compile-time error', '(b) Runtime exception',
    '(c) Logical error', '(d) No error occurs',
]

def build_corpus(n):
    base = _OOP*5 + _SE*3 + _AI*3 + _CYBER*2 + _SHORT*10
    # Marker combos are the single most error-prone pattern for TrOCR
    with_markers = []
    for m in _MARKERS:
        for s in _OOP + _SE + _AI:
            with_markers.append(f'{m} {s}')
        for s in _SHORT:
            with_markers.append(f'{m} {s}')
    pool = base + with_markers * 3
    random.shuffle(pool)
    times = math.ceil(n / len(pool))
    out = []
    for _ in range(times):
        t = pool[:]; random.shuffle(t); out.extend(t)
    return out[:n]

corpus = build_corpus(N_PAIRS * 4)
print(f'Corpus ready: {len(corpus):,} sentences')
print('Sample:', corpus[:3])

In [ ]:
# â”€â”€ Cell 5: Image renderer â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter

FONT_PATHS = [
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    '/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf',
    '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
]
FONT_SIZES = [22, 26, 30, 34, 38]  # more size diversity on A100
# Noise levels â€” more heavy-noise samples since A100 generates faster
_NOISE_LEVELS = [0.0, 0.0, 0.10, 0.20, 0.30, 0.45, 0.60, 0.75]

def _load_font(size):
    for p in FONT_PATHS:
        if os.path.exists(p):
            try: return ImageFont.truetype(p, size)
            except: pass
    return ImageFont.load_default()

def render_line(text, font_size=None, noise_level=0.0):
    if font_size is None:
        font_size = random.choice(FONT_SIZES)
    font = _load_font(font_size)
    dummy = Image.new('RGB', (1, 1))
    draw  = ImageDraw.Draw(dummy)
    bbox  = draw.textbbox((0, 0), text, font=font)
    tw, th = bbox[2]-bbox[0], bbox[3]-bbox[1]
    px, py = max(10, int(tw*0.06)), max(6, int(th*0.22))
    W, H = tw + 2*px, th + 2*py
    img  = Image.new('RGB', (W, H), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    ink  = (random.randint(0,20), random.randint(0,20), random.randint(0,35))
    draw.text((px, py), text, font=font, fill=ink)
    if noise_level > 0:
        arr = np.asarray(img, dtype=np.float32)
        if noise_level > 0.15:
            arr += np.random.normal(0, noise_level * 20, arr.shape)
            arr  = np.clip(arr, 0, 255)
        if noise_level > 0.35:
            mask = np.random.random(arr.shape[:2]) < noise_level * 0.022
            arr[mask] = np.random.choice([0, 255])
        img = Image.fromarray(arr.astype(np.uint8))
        if noise_level > 0.25:
            angle = random.uniform(-noise_level*3.0, noise_level*3.0)
            img   = img.rotate(angle, fillcolor=(255,255,255), expand=False)
        if noise_level > 0.45:
            img = img.filter(ImageFilter.GaussianBlur(radius=noise_level*1.1))
    return img.convert('RGB')

# Verify render
t = render_line('1) (b) Encapsulation hides state.', noise_level=0.4)
print(f'Render OK â€” size: {t.size}')

In [ ]:
# â”€â”€ Cell 6: Load TrOCR â€” same model PaperMind uses in production â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

TROCR_MODEL = 'microsoft/trocr-base-handwritten'
device = 'cuda'

print(f'Loading TrOCR ({TROCR_MODEL})â€¦')
trocr_proc  = TrOCRProcessor.from_pretrained(TROCR_MODEL)
trocr_model = VisionEncoderDecoderModel.from_pretrained(
    TROCR_MODEL, torch_dtype=torch.float16   # fp16 for TrOCR on A100
).eval().to(device)
print('TrOCR ready.')

@torch.inference_mode()
def trocr_batch(images):
    pixels = trocr_proc(images=images, return_tensors='pt').pixel_values.to(device)
    pixels = pixels.half()  # fp16
    out = trocr_model.generate(
        pixels, max_new_tokens=64, num_beams=1,
        output_scores=True, return_dict_in_generate=True
    )
    logprobs = trocr_model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
    pad    = trocr_proc.tokenizer.pad_token_id
    tokens = out.sequences[:, 1:]
    texts  = trocr_proc.batch_decode(out.sequences, skip_special_tokens=True)
    results = []
    for text, lp, tok in zip(texts, logprobs, tokens):
        mask    = tok != pad
        mean_lp = float(lp[mask].mean()) if mask.any() else -5.0
        results.append((text.strip(), math.exp(mean_lp)))
    return results

In [ ]:
# â”€â”€ Cell 7: GENERATE DATASET (50k pairs on A100, ~35 min) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import csv
from pathlib import Path

def levenshtein(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for ca in a:
        curr = [prev[0]+1]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j]+1, curr[-1]+1, prev[j-1]+(ca!=cb)))
        prev = curr
    return prev[-1]

pairs, corpus_iter, t0 = [], iter(corpus), time.time()

print(f'Generating {N_PAIRS:,} pairs (batch={TROCR_BATCH}, A100)â€¦')

while len(pairs) < N_PAIRS:
    batch_texts, batch_imgs, batch_noise = [], [], []
    for _ in range(TROCR_BATCH):
        try:    text = next(corpus_iter)
        except: corpus_iter = iter(corpus); text = next(corpus_iter)
        nl  = random.choice(_NOISE_LEVELS)
        img = render_line(text, font_size=random.choice(FONT_SIZES), noise_level=nl)
        batch_texts.append(text); batch_imgs.append(img); batch_noise.append(nl)

    results = trocr_batch(batch_imgs)

    for clean, (noisy, conf), nl in zip(batch_texts, results, batch_noise):
        ed = levenshtein(noisy, clean)
        if ed == 0 and random.random() > CLEAN_FRAC:
            continue
        pairs.append({'input': noisy, 'target': clean,
                      'edit_distance': ed, 'trocr_conf': round(conf,4),
                      'noise_level': round(nl,2)})

    n = len(pairs)
    if (n // 1000) > ((n - TROCR_BATCH) // 1000) or n >= N_PAIRS:
        el  = time.time()-t0
        eta = (N_PAIRS-n)/(n/el) if n>0 else 0
        print(f'  {n:>6,}/{N_PAIRS:,}  {el/60:.1f}m elapsed  ~{eta/60:.1f}m remaining')
    if len(pairs) >= N_PAIRS: break

pairs = pairs[:N_PAIRS]
random.shuffle(pairs)

perfect  = sum(1 for p in pairs if p['edit_distance']==0)
avg_ed   = sum(p['edit_distance'] for p in pairs)/len(pairs)
avg_conf = sum(p['trocr_conf']    for p in pairs)/len(pairs)
print(f'\n=== Dataset ===')
print(f'  Total   : {len(pairs):,}')
print(f'  Perfect : {perfect:,} ({100*perfect/len(pairs):.1f}%)')
print(f'  Avg ED  : {avg_ed:.2f}')
print(f'  Avg conf: {avg_conf:.3f}')
print(f'  Time    : {(time.time()-t0)/60:.1f} min')
print()
print('=== Sample errors ===')
shown=0
for p in pairs:
    if p['edit_distance']>0:
        print(f"  IN : {p['input']!r}")
        print(f"  OUT: {p['target']!r}  (ED={p['edit_distance']}  conf={p['trocr_conf']})")
        print()
        shown+=1
    if shown>=8: break

In [ ]:
# â”€â”€ Cell 8: Save CSVs to Drive â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
n       = len(pairs)
n_train = int(n*0.80); n_val = int(n*0.10)
splits  = {'train': pairs[:n_train],
            'val':   pairs[n_train:n_train+n_val],
            'test':  pairs[n_train+n_val:]}
fields  = ['input','target','edit_distance','trocr_conf','noise_level']
for name, rows in splits.items():
    path = Path(DRIVE_DATA_DIR)/f'{name}.csv'
    with open(path,'w',newline='',encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(rows)
    print(f'Saved {path} ({len(rows):,} rows)')
print('Dataset saved to Drive.')

In [ ]:
# â”€â”€ Cell 9: Free TrOCR VRAM â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Even with 40GB we want the full VRAM budget for byt5-base training.
del trocr_model, trocr_proc
torch.cuda.empty_cache()
import gc; gc.collect()
print(f'VRAM after TrOCR unload: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')
print('Ready for byt5-base training.')

In [ ]:
# â”€â”€ Cell 10: Build HuggingFace Dataset â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from datasets import Dataset, DatasetDict

to_hf = lambda rows: Dataset.from_dict({
    'input':  [r['input']  for r in rows],
    'target': [r['target'] for r in rows],
})
raw = DatasetDict({'train': to_hf(splits['train']),
                   'val':   to_hf(splits['val']),
                   'test':  to_hf(splits['test'])})
print(raw)

In [ ]:
# â”€â”€ Cell 11: Tokenise (ByT5 byte-level) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BYT5_MODEL)
print(f'Tokeniser: {type(tok).__name__}  vocab={tok.vocab_size} byte tokens')

def preprocess(examples):
    inp = tok(examples['input'],  max_length=MAX_LEN, padding='max_length', truncation=True)
    with tok.as_target_tokenizer():
        lbl = tok(examples['target'], max_length=MAX_LEN, padding='max_length', truncation=True)
    lbl['input_ids'] = [[(t if t!=tok.pad_token_id else -100) for t in seq] for seq in lbl['input_ids']]
    inp['labels'] = lbl['input_ids']
    return inp

tokenized = raw.map(preprocess, batched=True, remove_columns=['input','target'], desc='Tokenising')
print(tokenized)

In [ ]:
# â”€â”€ Cell 12: Load ByT5-base â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# byt5-base: 580M params â€” 2Ã— byt5-small.
# A100 40GB handles bs=64 with bf16 easily.
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    BYT5_MODEL, torch_dtype=torch.bfloat16   # bfloat16 â€” native A100 precision
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {BYT5_MODEL}  |  {n_params/1e6:.0f}M params')
print(f'VRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# â”€â”€ Cell 13: Metrics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import numpy as np

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    if preds.ndim == 3: preds = np.argmax(preds, axis=-1)
    dec_preds  = tok.batch_decode(preds, skip_special_tokens=True)
    labels_np  = np.where(labels!=-100, labels, tok.pad_token_id)
    dec_labels = tok.batch_decode(labels_np, skip_special_tokens=True)

    exact = sum(p.strip()==l.strip() for p,l in zip(dec_preds,dec_labels))/len(dec_preds)

    total_ed, total_len = 0, 0
    for p, l in zip(dec_preds, dec_labels):
        total_ed  += levenshtein(p.strip(), l.strip())
        total_len += max(len(l.strip()), 1)

    return {'exact_match': round(exact,4), 'cer': round(total_ed/total_len,4)}

print('Metrics: exact_match â†‘   CER â†“   (target: CER < 0.03 with byt5-base)')

In [ ]:
# â”€â”€ Cell 14: Trainer setup â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

args = Seq2SeqTrainingArguments(
    output_dir=DRIVE_CKPT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BYT5_TRAIN_BATCH,
    per_device_eval_batch_size=BYT5_EVAL_BATCH,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    save_total_limit=2,
    bf16=True,                   # bfloat16 â€” A100 native, more stable than fp16
    fp16=False,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    dataloader_num_workers=4,
    logging_steps=100,
    report_to='none',
)

collator = DataCollatorForSeq2Seq(tok, model=model, padding=True, pad_to_multiple_of=8)
trainer  = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['val'],
    tokenizer=tok,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
steps = len(tokenized['train']) // (BYT5_TRAIN_BATCH * max(GRAD_ACCUM_STEPS,1))
print(f'Trainer ready.  Train={len(tokenized["train"]):,}  Val={len(tokenized["val"]):,}')
print(f'Steps/epoch â‰ˆ {steps:,}   Total steps â‰ˆ {steps*EPOCHS:,}')

In [ ]:
# â”€â”€ Cell 15: TRAIN (~25â€“35 min on A100) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Watch the CER column each epoch â€” target < 0.03 by epoch 3.
# Best checkpoint (lowest val CER) auto-loads when done.
t_train = time.time()
result  = trainer.train()
print(f'Training done in {(time.time()-t_train)/60:.1f} min')
print(result.metrics)

In [ ]:
# â”€â”€ Cell 16: Save best checkpoint to Drive â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
trainer.save_model(DRIVE_CKPT_DIR)
tok.save_pretrained(DRIVE_CKPT_DIR)
print(f'Saved â†’ {DRIVE_CKPT_DIR}')
print('Files:', sorted(os.listdir(DRIVE_CKPT_DIR)))

In [ ]:
# â”€â”€ Cell 17: Test set evaluation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
test_res = trainer.evaluate(tokenized['test'])
em  = test_res.get('eval_exact_match', 0)
cer = test_res.get('eval_cer', 1)
print('=== Test Results (byt5-base, 50k pairs) ===')
print(f'  Exact Match : {em:.4f}  ({100*em:.1f}% of lines corrected perfectly)')
print(f'  CER         : {cer:.4f}  ({100*cer:.1f}% char error rate after correction)')
print(f'  Loss        : {test_res.get("eval_loss",0):.4f}')
print()
if   cer < 0.03: print('âœ“ Excellent â€” CER < 3%. Ready for production.')
elif cer < 0.05: print('~ Good â€” CER < 5%. Usable; consider +10k pairs or +1 epoch.')
else:            print('âœ— CER >= 5%. Inspect sample errors in Cell 18.')

In [ ]:
# â”€â”€ Cell 18: Qualitative â€” real PaperMind TrOCR errors â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Ground truth from storage_demo/answer_sheets/7/ocr.json
real_errors = [
    ('19 ( 0b ) .',                                 '1) (b).'),
    ('2 ) False',                                   '2) False'),
    ('3 ) Liskov . principle says objects of a',    '3) Liskov principle says objects of a'),
    ('program . Square extending . i Rectangle',    'program. Square extending Rectangle'),
    ('4 ) Encapsulation bundles data and methods',  '4) Encapsulation bundles data and methods'),
    ('using private access . It protects the data', 'using private access. It protects the data'),
    ('lnheritance aIIows a subclass',               'Inheritance allows a subclass'),
    ('0bject-oriented prograrnming',                'Object-oriented programming'),
    ('Q1 . (b)',                                    'Q1. (b)'),
    ('rn ethod overriding',                         'method overriding'),
    ('and reduces coupling between classes .',      'and reduces coupling between classes.'),
]

model.eval()
dev = next(model.parameters()).device
exact, improved, missed = 0, 0, 0
print('=== Qualitative check â€” real PaperMind OCR errors ===')
for noisy, expected in real_errors:
    inp = tok(noisy, return_tensors='pt', max_length=256, truncation=True)
    inp = {k: v.to(dev) for k,v in inp.items()}
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=256, num_beams=4)
    pred   = tok.decode(out[0], skip_special_tokens=True).strip()
    ed_in  = levenshtein(noisy, expected)
    ed_out = levenshtein(pred,  expected)
    if pred == expected:  tag = 'âœ“ EXACT   '; exact += 1
    elif ed_out < ed_in:  tag = '~ IMPROVED'; improved += 1
    else:                 tag = 'âœ— MISS    '; missed += 1
    print(f'{tag}  "{noisy}"')
    if pred != expected:
        print(f'           â†’ "{pred}"  (expected "{expected}")')
    print(f'           ED {ed_in} â†’ {ed_out}')
    print()

total = len(real_errors)
print(f'Result: {exact}/{total} exact  {improved}/{total} improved  {missed}/{total} missed')

In [ ]:
# â”€â”€ Cell 19: Integration steps â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=' * 65)
print('CHECKPOINT READY â€” integrate into PaperMind backend')
print('=' * 65)
print()
print(f'Model  : {BYT5_MODEL}')
print(f'Pairs  : {N_PAIRS:,}')
print(f'Test CER    : {test_res.get("eval_cer","?")}')
print(f'Test ExMatch: {test_res.get("eval_exact_match","?")}')
print()
print('1. Google Drive â†’ navigate to:')
print(f'   {DRIVE_CKPT_DIR}')
print('   Right-click folder â†’ Download (downloads as .zip)')
print()
print('2. Extract to:')
print('   C:\\PaperMind\\backend\\models\\byt5-correction\\')
print('   Must contain: config.json, tokenizer.json, model.safetensors')
print()
print('3. Add to C:\\PaperMind\\backend\\.env:')
print('   CORRECTION_MODEL_PATH=./models/byt5-correction')
print()
print('4. Restart backend â€” look for:')
print('   seq2seq:./models/byt5-correction')
print()
print('5. Re-process an answer sheet and verify corrected.json')
print('   "19 ( 0b ) ." should now read "1) (b)."')